# Stiff System: Custom Loss Combinations

Train non-uniform timesteps for the stiff LTI system (time constants 0.1s, 10s, 100s) using a fully configurable composite loss:
$$\mathcal{L} = w_{\text{ocp}} \cdot \mathcal{L}_{\text{ocp}} + \sum_i w_i \cdot \mathcal{L}_i$$

Set `loss_weights` to whichever regularizers you want to combine, with their fixed weights. Available losses: `L_IV`, `L_EQ`, `L_CPC`, `L_CSS`, `L_defect`, `L_dyn`, `L_equi`, `L_FI`, `L_SC`, `L_PWLH`, `L_SSD`.

In [ ]:
import sys, os
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', '..', 'src'))
from dimp.utils import init_matplotlib, get_colors
init_matplotlib()
colors = get_colors()

from stiff_sys_train import train_custom_loss
from stiff_sys_prob import T
from utils import plot_loss_results

## Configuration

Edit `loss_weights` and the training settings below to test different combinations.

In [ ]:
ocp_weight = 1.0
loss_weights = {"L_IV_sym": 0.0}
discretization = "zoh"     # "zoh" or "euler"
detach = "none"            # "none", "reg", or "all" (gradient detach mode)
n_steps = 40
n_epochs = 200
lr = 5e-2

tau_schedule = ("exp", 5.0, 0.03)

## Train

In [ ]:
sol_custom, history_custom, n_custom = train_custom_loss(
    loss_weights, n=n_steps, n_epochs=n_epochs, lr=lr,
    discretization=discretization, ocp_weight=ocp_weight,
    detach=detach, tau_schedule=tau_schedule,
)


## Plot results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6.4, 3.6), constrained_layout=True)

epochs = [h["epoch"] for h in history_custom]

axes[0].plot(epochs, [h["loss"] for h in history_custom], label="total")
axes[0].plot(epochs, [ocp_weight * h["loss_ocp"] for h in history_custom],
             label=f"{ocp_weight}·L_ocp", ls="--")
axes[0].set(xlabel="Epoch", ylabel="Loss", title="Loss curves")
axes[0].legend(fontsize=7)

for name, w in loss_weights.items():
    axes[1].plot(epochs, [w * h[f"loss_{name}"] for h in history_custom],
                 label=f"{w}·{name}")
axes[1].plot(epochs, [h["loss_reg_total"] for h in history_custom],
             label="reg total", ls="--", color="k")
axes[1].set(xlabel="Epoch", ylabel="Loss", title="Weighted regularizer losses")
axes[1].legend(fontsize=7)

label = f"{ocp_weight}·L_ocp + " + " + ".join(f"{w}·{n}" for n, w in loss_weights.items())
fig.suptitle(f"Custom: {label}", fontsize=11)
plt.show()

In [ ]:
result_custom = {"sol": sol_custom, "history": history_custom, "n": n_custom}
plot_loss_results(label, result_custom, results_dir=None, show=True)